# Phase 6e Wave 6 — Einstein-Cartan Extension (torsion from spin current)

**Lean module:** `lean/SKEFTHawking/EinsteinCartanExtension.lean`

**Python subpackage:** `src/einstein_cartan/`

**Paper:** `papers/paper43_einstein_cartan/paper_draft.tex`

**Wave 6 correctness-push:** at the natural microscopic point
(Λ_UV, N_f, α_EC) = (M_Pl, 16, 1) and cosmological-bath spin density,
predicted torsion |T_EC| ≃ 2×10⁻⁷⁷ GeV — ~46 orders of magnitude
below the Kostelecky-Russell-Tasson bound 10⁻³¹ GeV (PRL 100, 111102
(2008)). The Einstein-Cartan extension of the ADW emergent-gravity
programme is observationally consistent at all natural parameter
points.

## §1. Microscopic torsion-amplitude prediction

Hehl-style EC theory gives the algebraic Cartan torsion equation
`T = G_N · S` for the spin-current source. In the ADW formulation
`G_N = G_N^emerg = α_EC · G_N_sakharov`, so the scalar Wave-6
amplitude is

$$|T_{\mathrm{EC}}|(\Lambda_{\mathrm{UV}}, N_f, \alpha_{\mathrm{EC}}, n_{\mathrm{spin}}) = \alpha_{\mathrm{EC}} \cdot \frac{12\pi}{N_f \Lambda_{\mathrm{UV}}^2} \cdot n_{\mathrm{spin}}.$$

Lean theorems:
- `torsionAmplitude_pos` — positivity (rules out vanishing prediction)
- `torsionAmplitude_eq_alpha_times_g_n_from_a2_times_n` — linear in α_EC
- `torsionAtCosmologicalBackground_pos_at_natural_params` — non-trivial at natural params

In [1]:
from src.core.constants import EINSTEIN_CARTAN_PARAMS
from src.core.formulas import (
    torsion_amplitude_ec,
    torsion_amplitude_at_cosmological_background,
    G_N_from_seeley_dewitt,
)
import math
M_Pl = 1.221e19
N_f = EINSTEIN_CARTAN_PARAMS['N_F_SM_DIRAC']
alpha_EC = EINSTEIN_CARTAN_PARAMS['ALPHA_EC_CALIBRATED']
n_s = EINSTEIN_CARTAN_PARAMS['COSMOLOGICAL_SPIN_DENSITY_GEV3']
T_EC = torsion_amplitude_at_cosmological_background(M_Pl, N_f, alpha_EC)
G_N = G_N_from_seeley_dewitt(M_Pl, N_f)
print(f'G_N_from_a2(M_Pl, N_f=16) = {G_N:.3e} GeV⁻²')
print(f'cosmological n_s         = {n_s:.3e} GeV³')
print(f'|T_EC| at natural params = {T_EC:.3e} GeV')

G_N_from_a2(M_Pl, N_f=16) = 1.580e-38 GeV⁻²
cosmological n_s         = 1.300e-39 GeV³
|T_EC| at natural params = 2.055e-77 GeV


## §2. Decision-Gate-style observational-bound passage

The tightest published cosmic-axial-torsion bound is
Kostelecky-Russell-Tasson (PRL 100, 111102 (2008)):
$T_{\mathrm{Kost}} = 10^{-31}$ GeV.

The cross-channel-independent Hughes-Drever bound (Lammerzahl PRD
64, 084014 (2001)) is looser by 100×: $T_{\mathrm{HD}} = 10^{-29}$ GeV.

Wave 6 main correctness-push (Lean theorem
`torsionAtCosmologicalBackground_at_planck_natural_below_kostelecky`):
the predicted amplitude sits ~46 orders of magnitude below Kostelecky
at the natural microscopic point.

In [2]:
from src.einstein_cartan import torsion_observational_verdict
v = torsion_observational_verdict(M_Pl, N_f, alpha_EC)
print(f'|T_EC|                    = {v.amplitude_gev:.3e} GeV')
print(f'Kostelecky satisfied      : {v.kostelecky_satisfied}')
print(f'Hughes-Drever satisfied   : {v.hughes_drever_satisfied}')
print(f'log10(Kost/|T_EC|) margin = {v.log10_margin_kostelecky:.2f} decades')
print(f'verdict                   : {v.verdict_label}')

|T_EC|                    = 2.055e-77 GeV
Kostelecky satisfied      : True
Hughes-Drever satisfied   : True
log10(Kost/|T_EC|) margin = 45.69 decades
verdict                   : torsion_below_bound


## §3. Match residual biconditional (Wave 6 expression of
Decision Gate E.2)

The EC match residual

$$\delta T_{\mathrm{EC}}(\Lambda_{\mathrm{UV}}, N_f, \alpha_{\mathrm{EC}}, n_{\mathrm{spin}}) = (\alpha_{\mathrm{EC}} - 1)\cdot G_N^{\mathrm{Sakharov}}(\Lambda_{\mathrm{UV}}, N_f) \cdot n_{\mathrm{spin}}$$

vanishes iff α_EC = 1 under positive (Λ_UV, N_f, n_spin). Forward
direction uses Wave 1's `G_N_from_a2_pos` (positivity) and the
non-vanishing `n_spin ≠ 0` to flip two `mul_eq_zero` splits.

Lean: `ecResidual_eq_zero_iff_alpha_unity`.

In [3]:
from src.einstein_cartan import ec_residual_at_point
for alpha in [0.5, 0.99, 1.0, 1.01, 1.5, 2.0]:
    r = ec_residual_at_point(M_Pl, N_f, alpha)
    flag = '✓' if r.match_holds else '✗'
    print(f'α_EC={alpha:5.2f}  residual={r.residual_gev:+.3e}  bundle:{flag}')

α_EC= 0.50  residual=-1.027e-77  bundle:✗
α_EC= 0.99  residual=-2.055e-79  bundle:✗
α_EC= 1.00  residual=+0.000e+00  bundle:✓
α_EC= 1.01  residual=+2.055e-79  bundle:✗
α_EC= 1.50  residual=+1.027e-77  bundle:✗
α_EC= 2.00  residual=+2.055e-77  bundle:✗


## §4. Parameter scans — log-log torsion amplitude

At α_EC = 1 (calibration), |T_EC| ∝ 1/(N_f · Λ_UV²) · n_spin. The
linear scan over Λ_UV (TeV → M_Pl) at N_f benchmarks demonstrates
that even at TeV-scale cutoffs, |T_EC| sits well below all bounds.

In [4]:
from src.einstein_cartan import torsion_scan_over_lambda_uv
import numpy as np
for N_f_val in [1, 4, 16, 100]:
    scan = torsion_scan_over_lambda_uv(
        N_f_val, alpha_ec=1.0,
        lambda_uv_values=[1e3, 1e10, 1e16, 1.221e19],
    )
    print(f'N_f={N_f_val:3d}: ', end='')
    for p in scan:
        print(f'  Λ={p.lambda_uv_gev:.0e}→{p.amplitude_gev:.2e}', end='')
    print()

N_f=  1:   Λ=1e+03→4.90e-44  Λ=1e+10→4.90e-58  Λ=1e+16→4.90e-70  Λ=1e+19→3.29e-76
N_f=  4:   Λ=1e+03→1.23e-44  Λ=1e+10→1.23e-58  Λ=1e+16→1.23e-70  Λ=1e+19→8.22e-77
N_f= 16:   Λ=1e+03→3.06e-45  Λ=1e+10→3.06e-59  Λ=1e+16→3.06e-71  Λ=1e+19→2.05e-77
N_f=100:   Λ=1e+03→4.90e-46  Λ=1e+10→4.90e-60  Λ=1e+16→4.90e-72  Λ=1e+19→3.29e-78


## §5. Bundled tracked-Prop predicate

The Wave 6 EC extension is bundled as a 3-conjunct Prop:

$$H = \{ \delta T_{\mathrm{EC}} = 0 \} \wedge \{ |T_{\mathrm{EC}}| > 0 \} \wedge \{ |T_{\mathrm{EC}}| < T_{\mathrm{Kost}} \}$$

Each conjunct probes a structurally distinct channel:
1. Calibration (α_EC = 1)
2. Non-vanishing prediction (rules out trivial passing)
3. Observational-bound passage (correctness-push)

Lean witnesses:
- `dirac_H_EinsteinCartanExtensionHolds_at_alpha_one` — bundle holds at α=1
- `perturbed_alpha_not_H_EinsteinCartanExtensionHolds` — falsified for α≠1

In [5]:
from src.core.formulas import einstein_cartan_extension_holds
for alpha in [0.5, 1.0, 1.5, 2.0]:
    h = einstein_cartan_extension_holds(M_Pl, N_f, alpha)
    flag = '✓' if h else '✗'
    print(f'  α_EC={alpha:.1f}  H_EC = {flag}')

  α_EC=0.5  H_EC = ✗
  α_EC=1.0  H_EC = ✓
  α_EC=1.5  H_EC = ✗
  α_EC=2.0  H_EC = ✗


## §6. Visualization — fig_torsion_obs_bound

Two-panel figure:
- Left: log-log |T_EC|(Λ_UV) curves at fixed N_f vs Kostelecky/Hughes-Drever
- Right: 2D headroom heatmap log10(Kost/|T_EC|) over (Λ_UV, N_f) at α_EC=1

In [6]:
from src.core.visualizations import fig_torsion_obs_bound
fig = fig_torsion_obs_bound()
fig.show()